In [22]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from plotly_football_pitch import make_pitch_figure, PitchDimensions

def short_match_id_from_source_file(source_file: str) -> str:
    """
    '1_Roma_Bologna_cl0nfg...csv' -> 'ROM-BOL'
    """
    name = Path(source_file).stem
    parts = name.split("_")

    if len(parts) >= 3:
        home = parts[1]
        away = parts[2]
    else:
        return "UNK-UNK"

    def abbr(s: str) -> str:
        s = re.sub(r"[^A-Za-z]", "", s)
        s = s[:3].upper() if len(s) >= 3 else s.upper()
        return s

    return f"{abbr(home)}-{abbr(away)}"

# ----------------------------
# CONFIG
# ----------------------------
RESULT_DIR = (
    Path.home()
    / "Documents"
    / "Master Degree in Sport Analytics"
    / "2. Management and Architecture of Sports Database"
    / "selenium wire"
    / "Result_SerieA_25_26"
)

ONLY_OPEN_PLAY = True
HIGH_REGAIN_X_MIN = 66.7

HIGH_REGAIN_TYPES = ["Ball recovery"] #"Interception", "Tackle"]
SHOT_TYPES = ["Goal", "Saved Shot", "Miss", "Post"]


# ----------------------------
# 1) Find + Load
# ----------------------------
def find_event_files(root: Path):
    files = sorted(root.rglob("*.csv"))
    files = [f for f in files if "-checkpoint" not in f.name.lower()]
    if not files:
        raise FileNotFoundError(f"Nessun CSV trovato in {root}")
    return files

def load_events(paths):
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        df["source_file"] = p.name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


# ----------------------------
# 2) Standardize
# ----------------------------
def standardize(df):
    out = df.copy()
    out = out.rename(columns={
        "event": "type",
        "period_id": "period",
        "time_min": "minute",
        "time_sec": "second",
        "team_name": "team",
        "player_name": "player",
        "represented_qualifiers": "qualifiers_raw",
        "non_represented_qualifiers": "qualifiers_raw_2",
    })

    for c in ["x", "y", "minute", "second", "period", "outcome", "match_id", "event_id", "general_id"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    if "x" in out.columns:
        out["x"] = out["x"].clip(0, 100)
    if "y" in out.columns:
        out["y"] = out["y"].clip(0, 100)

    sort_cols = [c for c in ["match_id", "period", "minute", "second", "event_id"] if c in out.columns]
    if sort_cols:
        out = out.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    return out


# ----------------------------
# 3) Open play tagging
# ----------------------------
def qtext(df):
    q1 = df.get("qualifiers_raw", pd.Series("", index=df.index)).fillna("").astype(str)
    q2 = df.get("qualifiers_raw_2", pd.Series("", index=df.index)).fillna("").astype(str)
    return (q1 + " ; " + q2).str.lower()

def add_phase_of_play(ev):
    qt = qtext(ev)

    is_corner  = qt.str.contains(r"\bcorner\b")
    is_fk      = qt.str.contains(r"\bfree kick\b") | qt.str.contains(r"\bdirect free\b") | qt.str.contains(r"\bindirect free\b")
    is_throwin = qt.str.contains(r"\bthrow in\b") | qt.str.contains(r"\bthrow-in\b")
    is_pen     = qt.str.contains(r"\bpenalty\b")
    is_gk      = qt.str.contains(r"\bgoal kick\b") | qt.str.contains(r"\bkick from hands\b")
    is_out     = qt.str.contains(r"\bout of play\b") | (ev.get("type", pd.Series("", index=ev.index)) == "Out")

    restart_type = np.select(
        [is_pen, is_corner, is_fk, is_throwin, is_gk, is_out],
        ["penalty", "corner", "free_kick", "throw_in", "goal_kick", "out_of_play"],
        default="none"
    )

    is_admin = ev.get("type", pd.Series("", index=ev.index)).isin(
        ["Start","End","Team setp up","Resume","Formation change","Player on","Player Off","Substitution"]
    )
    is_set_piece = np.isin(restart_type, ["penalty","corner","free_kick","throw_in","goal_kick"])
    is_open_play = (~is_set_piece) & (~is_admin) & (~is_out)

    ev["phase_of_play"] = np.select([is_set_piece, is_open_play], ["set_piece","open_play"], default="unknown")
    return ev


# ----------------------------
# 4) Normalize attack to the right (x flip) using shots
# ----------------------------
def normalize_attack_right_using_shots(df_team):
    out = df_team.copy()
    out["x"] = pd.to_numeric(out["x"], errors="coerce")
    out["y"] = pd.to_numeric(out["y"], errors="coerce")

    is_shot = out["type"].isin(SHOT_TYPES)

    # Qui serve una chiave partita: prova match_id, altrimenti contestant_id, altrimenti general_id
    match_key = infer_match_key_col(out)
    keys = [match_key, "period"]

    shot_count = out.loc[is_shot].groupby(keys).size()
    shot_mean_x = out.loc[is_shot].groupby(keys)["x"].mean()

    flip = (shot_count >= 3) & (shot_mean_x < 50)

    flip_flags = flip.reset_index()
    flip_flags.columns = [match_key, "period", "flip"]

    out = out.merge(flip_flags, on=[match_key, "period"], how="left")
    out["flip"] = out["flip"].map(lambda x: bool(x) if pd.notna(x) else False)

    out.loc[out["flip"], "x"] = 100 - out.loc[out["flip"], "x"]
    return out.drop(columns=["flip"])


# ----------------------------
# 5) Match id / label robusti
# ----------------------------
def infer_match_key_col(df: pd.DataFrame) -> str:
    """
    Trova una colonna che identifichi la partita.
    Nel tuo screenshot sembra che 'contestant_id' o 'general_id' siano la chiave partita.
    """
    candidates = [
        "match_id",
        "contestant_id",   # nel tuo df c'è
        "fixture_id",
        "game_id",
        "general_id",      # nel tuo df c'è
        "source_file"      # fallback: un file = una partita (spesso vero nel tuo setup)
    ]
    for c in candidates:
        if c in df.columns:
            # deve avere almeno 1 valore non nullo
            if df[c].notna().any():
                return c
    # ultimissimo fallback
    return "source_file"


def build_match_label(df: pd.DataFrame, match_key_col: str) -> pd.Series:
    """
    Crea un'etichetta partita leggibile.
    Se hai home/away nel df, qui possiamo migliorare ulteriormente.
    Al momento: usa competition + match_key.
    """
    comp = None
    for c in ["competition_known_name", "competition_name", "competition"]:
        if c in df.columns:
            comp = df[c].fillna("").astype(str)
            break

    key = df[match_key_col].astype(str)

    if comp is not None and (comp.str.len() > 0).any():
        return comp.where(comp.str.len() > 0, "match") + " | " + key
    return "match | " + key


# ----------------------------
# 6) Build df_high_regains
# ----------------------------
def build_high_regains_df(team_str: str, match_value=None):
    paths = find_event_files(RESULT_DIR)
    raw = load_events(paths)
    ev = standardize(raw)
    ev = add_phase_of_play(ev)

    # filtro squadra
    df_team = ev[ev["team"].astype(str).str.contains(team_str, case=False, na=False)].copy()

    if df_team.empty:
        raise ValueError(f"Nessun evento trovato per team='{team_str}'")

    # match key & filtro match opzionale
    match_key_col = infer_match_key_col(df_team)
    if match_value is not None:
        df_team = df_team[df_team[match_key_col].astype(str) == str(match_value)].copy()

    # open play
    if ONLY_OPEN_PLAY:
        df_team = df_team[df_team["phase_of_play"] == "open_play"].copy()

    if df_team.empty:
        raise ValueError(f"Nessun evento trovato dopo i filtri per team='{team_str}' e match='{match_value}'")

    # normalizza direzione
    df_team = normalize_attack_right_using_shots(df_team)

    # high regains types
    df_hr = df_team[df_team["type"].isin(HIGH_REGAIN_TYPES)].copy()

    # tackle riusciti
    if "outcome" in df_hr.columns:
        is_tackle = df_hr["type"].eq("Tackle")
        df_hr = df_hr[~is_tackle | (df_hr["outcome"] == 1)].copy()

    # x threshold
    df_hr = df_hr[df_hr["x"] >= HIGH_REGAIN_X_MIN].copy()

    # colonne per hover
    df_hr["match_key"] = df_hr[match_key_col].astype(str)
    df_hr["match_label"] = build_match_label(df_hr, match_key_col)

        # ---- ADD: match_short tipo ROM-BOL dal nome file ----
    if "source_file" in df_hr.columns:
        df_hr["match_short"] = df_hr["source_file"].astype(str).map(short_match_id_from_source_file)
    else:
        df_hr["match_short"] = "UNK-UNK"

    return df_hr


# ----------------------------
# 7) Plotly pitch + heatmap + hover points
# ----------------------------
def plot_high_regains_on_pitch_points_only(df_hr: pd.DataFrame, title: str):
    if df_hr.empty:
        fig = go.Figure()
        fig.update_layout(title=f"{title} — nessun evento")
        return fig

    dimensions = PitchDimensions()
    fig = make_pitch_figure(dimensions)

    pitch_L = getattr(dimensions, "pitch_length", 105)
    pitch_W = getattr(dimensions, "pitch_width", 68)

    # Opta 0-100 -> pitch dims (es. 120x80)
    x_plot = df_hr["x"].astype(float).to_numpy() / 100.0 * pitch_L
    y_plot = df_hr["y"].astype(float).to_numpy() / 100.0 * pitch_W

    # colonne per hover
    player_col = "player" if "player" in df_hr.columns else "player_name" if "player_name" in df_hr.columns else None
    if player_col is None:
        df_hr = df_hr.copy()
        df_hr["player"] = ""
        player_col = "player"

    if "minute" not in df_hr.columns:
        df_hr = df_hr.copy()
        df_hr["minute"] = np.nan
    if "type" not in df_hr.columns:
        df_hr = df_hr.copy()
        df_hr["type"] = ""

    if "match_short" not in df_hr.columns:
        df_hr = df_hr.copy()
        df_hr["match_short"] = "UNK-UNK"

    if "match_key" not in df_hr.columns:
        df_hr = df_hr.copy()
        df_hr["match_key"] = ""

    if "second" not in df_hr.columns:
        df_hr = df_hr.copy()
        df_hr["second"] = np.nan

    customdata = np.stack([
        df_hr["match_short"].astype(str).to_numpy(),  # ROM-BOL
        df_hr["match_key"].astype(str).to_numpy(),
        df_hr["minute"].to_numpy(),
        df_hr[player_col].astype(str).to_numpy(),
        df_hr["type"].astype(str).to_numpy(),
        df_hr["second"].to_numpy(),
    ], axis=-1)

    fig.add_trace(go.Scattergl(
        x=x_plot,
        y=y_plot,
        mode="markers",
        marker=dict(
            size=8,
            color="yellow",   # contrasto forte sul pitch scuro
            opacity=0.85
        ),
        name="High regains",
        customdata=customdata,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"               # ROM-BOL
            "Minuto: %{customdata[2]}<br>"
            "Secondo: %{customdata[5]}<br>"
            "Giocatore: %{customdata[3]}<br>"
            "Tipo: %{customdata[4]}<br>"
            "x: %{x:.2f} — y: %{y:.2f}<br>"
            "Match key: %{customdata[1]}"
            "<extra></extra>"
        )
    ))

    fig.update_layout(title=title, showlegend=False)
    return fig


# ----------------------------
# 8) RUN
# ----------------------------
team_str = "Inter"
df_high_regains = build_high_regains_df(team_str=team_str, match_value=None)

#print("High regains:", len(df_high_regains))
#display(df_high_regains[["source_file","match_short","match_key","minute","second","player","type","x","y"]].head(10))

fig = plot_high_regains_on_pitch_points_only(
    df_high_regains,
    title=f"{team_str} — High Regains (x ≥ {HIGH_REGAIN_X_MIN}, Open Play={ONLY_OPEN_PLAY})"
)
fig.show()

In [23]:
# =============================================================================
# 9) LEAGUE TABLE — High Regains (Serie A)
# =============================================================================

# 1) Load all events (una sola volta)
paths = find_event_files(RESULT_DIR)
raw = load_events(paths)
ev = standardize(raw)
ev = add_phase_of_play(ev)

# 2) Open play only (coerente con il notebook)
if ONLY_OPEN_PLAY:
    ev = ev[ev["phase_of_play"] == "open_play"].copy()

# 3) Normalizza direzione per OGNI squadra
dfs = []
for team in ev["team"].dropna().unique():
    df_team = ev[ev["team"] == team].copy()
    if not df_team.empty:
        df_team = normalize_attack_right_using_shots(df_team)
        dfs.append(df_team)

ev_norm = pd.concat(dfs, ignore_index=True)

# 4) Filtra High Regains
df_hr = ev_norm[ev_norm["type"].isin(HIGH_REGAIN_TYPES)].copy()

# tackle riusciti
if "outcome" in df_hr.columns:
    is_tackle = df_hr["type"].eq("Tackle")
    df_hr = df_hr[~is_tackle | (df_hr["outcome"] == 1)].copy()

# soglia ultima trequarti offensiva
df_hr = df_hr[df_hr["x"] >= HIGH_REGAIN_X_MIN].copy()

# 5) League table
league_table = (
    df_hr
    .groupby("team", as_index=False)
    .size()
    .rename(columns={"size": "high_regains"})
    .sort_values("high_regains", ascending=False)
    .reset_index(drop=True)
)

league_table.index += 1  # ranking 1-based

display(league_table)

,team,high_regains
1,FC Internazionale Milano,133
2,Juventus FC,131
3,AS Roma,120
4,Genoa CFC,106
5,SSC Napoli,105
6,US Lecce,100
7,Atalanta Bergamasca Calcio,98
8,Udinese Calcio,93
9,Bologna FC 1909,92
10,AC Milan,89


In [24]:
# ----------------------------
# 9) LINK: High regains -> first shot within 30s (same match_key)
# ----------------------------
def add_t_abs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tempo assoluto in secondi nel match: (period-1)*45*60 + minute*60 + second
    """
    out = df.copy()
    out["period"] = pd.to_numeric(out.get("period", np.nan), errors="coerce").fillna(1)
    out["minute"] = pd.to_numeric(out.get("minute", np.nan), errors="coerce")
    out["second"] = pd.to_numeric(out.get("second", np.nan), errors="coerce").fillna(0)
    out["t_abs"] = (out["period"] - 1) * 45 * 60 + out["minute"] * 60 + out["second"]
    return out


def build_high_regains_to_shots_df(team_str: str, window_sec: int = 30, match_value=None):
    """
    Usa la stessa pipeline della tua build_high_regains_df:
    - carica eventi
    - standardize + open play tagging
    - filtro team
    - filtro open play
    - normalizza attacco a destra
    - high regains (types + x>=threshold + tackle success)
    + collega il primo shot/goal entro window_sec nello stesso match_key.
    Ritorna un df "linked" con colonne del regain + colonne shot_*.
    """
    paths = find_event_files(RESULT_DIR)
    raw = load_events(paths)
    ev = standardize(raw)
    ev = add_phase_of_play(ev)

    # filtro squadra
    df_team = ev[ev["team"].astype(str).str.contains(team_str, case=False, na=False)].copy()
    if df_team.empty:
        raise ValueError(f"Nessun evento trovato per team='{team_str}'")

    # match key & filtro match opzionale
    match_key_col = infer_match_key_col(df_team)
    if match_value is not None:
        df_team = df_team[df_team[match_key_col].astype(str) == str(match_value)].copy()

    # open play (identico al tuo)
    if ONLY_OPEN_PLAY:
        df_team = df_team[df_team["phase_of_play"] == "open_play"].copy()

    if df_team.empty:
        raise ValueError(f"Nessun evento trovato dopo i filtri per team='{team_str}' e match='{match_value}'")

    # normalizza direzione (identico al tuo)
    df_team = normalize_attack_right_using_shots(df_team)

    # aggiungi tempo assoluto
    df_team = add_t_abs(df_team)

    # ---- High regains (identico al tuo) ----
    df_hr = df_team[df_team["type"].isin(HIGH_REGAIN_TYPES)].copy()

    if "outcome" in df_hr.columns:
        is_tackle = df_hr["type"].eq("Tackle")
        df_hr = df_hr[~is_tackle | (df_hr["outcome"] == 1)].copy()

    df_hr = df_hr[df_hr["x"] >= HIGH_REGAIN_X_MIN].copy()

    # ---- Shots (nello stesso df_team) ----
    df_sh = df_team[df_team["type"].isin(SHOT_TYPES)].copy()

    # tieni solo righe con chiavi ok
    df_hr = df_hr.dropna(subset=[match_key_col, "t_abs", "x", "y"])
    df_sh = df_sh.dropna(subset=[match_key_col, "t_abs", "x", "y"])

    # match labels come nel tuo
    df_hr["match_key"] = df_hr[match_key_col].astype(str)
    df_hr["match_label"] = build_match_label(df_hr, match_key_col)
    if "source_file" in df_hr.columns:
        df_hr["match_short"] = df_hr["source_file"].astype(str).map(short_match_id_from_source_file)
    else:
        df_hr["match_short"] = "UNK-UNK"

    # se non ci sono shots, ritorna vuoto
    if df_hr.empty or df_sh.empty:
        return df_hr.iloc[0:0].copy()

    # ordina shots per match e tempo
    df_sh = df_sh.sort_values([match_key_col, "t_abs"], kind="mergesort").reset_index(drop=True)

    # indicizza shots per match_key -> array tempi e indice
    shots_by_match = {}
    for mk, g in df_sh.groupby(match_key_col, sort=False):
        shots_by_match[str(mk)] = g

    # collega ogni regain al primo shot successivo entro window_sec (stesso match_key)
    linked_rows = []

    df_hr_sorted = df_hr.sort_values([match_key_col, "t_abs"], kind="mergesort").reset_index(drop=True)

    for mk, g_hr in df_hr_sorted.groupby(match_key_col, sort=False):
        mk_str = str(mk)
        g_sh = shots_by_match.get(mk_str)
        if g_sh is None or g_sh.empty:
            continue

        shot_times = g_sh["t_abs"].to_numpy()
        hr_times = g_hr["t_abs"].to_numpy()

        # primo shot dopo ciascun regain
        idx = np.searchsorted(shot_times, hr_times, side="right")
        valid = idx < len(shot_times)
        if not valid.any():
            continue

        g_hr_v = g_hr.iloc[np.where(valid)[0]].copy()
        g_sh_v = g_sh.iloc[idx[valid]].copy()

        dt = g_sh_v["t_abs"].to_numpy() - g_hr_v["t_abs"].to_numpy()
        within = dt <= window_sec
        if not within.any():
            continue

        g_hr_v = g_hr_v.iloc[np.where(within)[0]].copy()
        g_sh_v = g_sh_v.iloc[np.where(within)[0]].copy()
        g_hr_v["dt_to_shot_sec"] = dt[within]

        # aggiungi colonne shot_*
        g_hr_v["shot_minute"] = g_sh_v["minute"].values
        g_hr_v["shot_second"] = g_sh_v["second"].values
        g_hr_v["shot_type"]   = g_sh_v["type"].astype(str).values
        g_hr_v["shot_player"] = g_sh_v["player"].astype(str).values if "player" in g_sh_v.columns else ""
        g_hr_v["shot_x"]      = g_sh_v["x"].astype(float).values
        g_hr_v["shot_y"]      = g_sh_v["y"].astype(float).values

        linked_rows.append(g_hr_v)

    if not linked_rows:
        return df_hr.iloc[0:0].copy()

    return pd.concat(linked_rows, ignore_index=True)


# ----------------------------
# 10) Plot: regains + shots + goals + lines
# ----------------------------
def plot_high_regains_to_shots(df_linked: pd.DataFrame, title: str):
    if df_linked.empty:
        fig = go.Figure()
        fig.update_layout(title=f"{title} — nessun regain→shot entro window")
        return fig

    dimensions = PitchDimensions()
    fig = make_pitch_figure(dimensions)

    pitch_L = getattr(dimensions, "pitch_length", 105)
    pitch_W = getattr(dimensions, "pitch_width", 68)

    # Convert Opta 0-100 -> 105x68
    x_r = df_linked["x"].astype(float).to_numpy() / 100.0 * pitch_L
    y_r = df_linked["y"].astype(float).to_numpy() / 100.0 * pitch_W
    x_s = df_linked["shot_x"].astype(float).to_numpy() / 100.0 * pitch_L
    y_s = df_linked["shot_y"].astype(float).to_numpy() / 100.0 * pitch_W

    # Ensure hover cols exist
    if "match_short" not in df_linked.columns:
        df_linked = df_linked.copy()
        df_linked["match_short"] = "UNK-UNK"
    if "player" not in df_linked.columns:
        df_linked = df_linked.copy()
        df_linked["player"] = ""
    if "second" not in df_linked.columns:
        df_linked = df_linked.copy()
        df_linked["second"] = np.nan

    # ---- Lines regain -> shot (no hover) ----
    for i in range(len(df_linked)):
        fig.add_trace(go.Scattergl(
            x=[x_r[i], x_s[i]],
            y=[y_r[i], y_s[i]],
            mode="lines",
            line=dict(width=1, color="rgba(255,255,255,0.35)"),
            hoverinfo="skip",
            showlegend=False
        ))

    # ---- Recoverers (circles) ----
    custom_r = np.stack([
        df_linked["match_short"].astype(str).to_numpy(),
        df_linked["minute"].to_numpy(),
        df_linked["second"].to_numpy(),
        df_linked["player"].astype(str).to_numpy(),
        df_linked["type"].astype(str).to_numpy(),
        df_linked["dt_to_shot_sec"].astype(float).to_numpy(),
        df_linked["x"].astype(float).to_numpy(),   # original 0-100 for hover if you want
        df_linked["y"].astype(float).to_numpy(),
    ], axis=-1)

    fig.add_trace(go.Scattergl(
        x=x_r,
        y=y_r,
        mode="markers",
        marker=dict(size=8, color="yellow", opacity=0.9),
        name="High regain",
        customdata=custom_r,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Min: %{customdata[1]} — Sec: %{customdata[2]}<br>"
            "Player: %{customdata[3]}<br>"
            "Type: %{customdata[4]}<br>"
            "x: %{customdata[6]:.2f} — y: %{customdata[7]:.2f}<br>"
            "→ Shot in: %{customdata[5]:.1f}s"
            "<extra></extra>"
        )
    ))

    # ---- Shooters: split goals vs non-goals ----
    shot_type_l = df_linked["shot_type"].astype(str).str.strip().str.lower()
    is_goal = shot_type_l.eq("goal")

    # Hover shooters
    custom_s = np.stack([
        df_linked["match_short"].astype(str).to_numpy(),
        df_linked["shot_minute"].to_numpy(),
        df_linked["shot_second"].to_numpy(),
        df_linked["shot_player"].astype(str).to_numpy(),
        df_linked["shot_type"].astype(str).to_numpy(),
        df_linked["shot_x"].astype(float).to_numpy(),
        df_linked["shot_y"].astype(float).to_numpy(),
    ], axis=-1)

    # Non-goal shots: triangles
    if (~is_goal).any():
        fig.add_trace(go.Scattergl(
            x=x_s[~is_goal],
            y=y_s[~is_goal],
            mode="markers",
            marker=dict(size=9, symbol="triangle-up", color="deepskyblue", opacity=0.9),
            name="Shot",
            customdata=custom_s[~is_goal],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Min: %{customdata[1]} — Sec: %{customdata[2]}<br>"
                "Shooter: %{customdata[3]}<br>"
                "Event: %{customdata[4]}<br>"
                "x: %{customdata[5]:.2f} — y: %{customdata[6]:.2f}"
                "<extra></extra>"
            )
        ))

    # Goals: stars
    if is_goal.any():
        fig.add_trace(go.Scattergl(
            x=x_s[is_goal],
            y=y_s[is_goal],
            mode="markers",
            marker=dict(size=11, symbol="star", color="lime", opacity=0.95),
            name="Goal",
            customdata=custom_s[is_goal],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Min: %{customdata[1]} — Sec: %{customdata[2]}<br>"
                "Scorer: %{customdata[3]}<br>"
                "Event: %{customdata[4]}<br>"
                "x: %{customdata[5]:.2f} — y: %{customdata[6]:.2f}"
                "<extra></extra>"
            )
        ))

    fig.update_layout(title=title, showlegend=False)
    return fig


# ----------------------------
# 11) RUN
# ----------------------------
team_str = "Inter"
df_linked = build_high_regains_to_shots_df(team_str=team_str, window_sec=15, match_value=None)

fig = plot_high_regains_to_shots(
    df_linked,
    title=f"{team_str} — High Regains (Open Play, x≥{HIGH_REGAIN_X_MIN}) → Shot/Goal ≤15s"
)
fig.show()


In [25]:
df_linked = build_high_regains_to_shots_df(team_str="Inter", window_sec=15)
print(df_linked)

    general_id  event_id           type  type_id  period  minute  second  \
0   2871921913        40  Ball recovery       49       1       2       1   
1   2878441749        41  Ball recovery       49       1       3      12   
2   2866168927        43  Ball recovery       49       1       3      12   
3   2890045505        57  Ball recovery       49       1       3      20   
4   2849766989        38  Ball recovery       49       1       3      21   
..         ...       ...            ...      ...     ...     ...     ...   
92  2871988673       845  Ball recovery       49       2      84      18   
93  2852502443       844  Ball recovery       49       2      84      24   
94  2866286921       874  Ball recovery       49       2      85      31   
95  2900414377       892  Ball recovery       49       2      88      27   
96  2888100433      1055  Ball recovery       49       2      97       3   

                contestant_id                      team team_code  ...  \
0   3vo5mpj7c

In [26]:
df_linked.head()

,general_id,event_id,type,type_id,period,minute,second,contestant_id,team,team_code,...,match_key,match_label,match_short,dt_to_shot_sec,shot_minute,shot_second,shot_type,shot_player,shot_x,shot_y
0,2871921913,40,Ball recovery,49,1,2,1,3vo5mpj7catp66nrwwqiuhuup,FC Internazionale Milano,INT,...,3vo5mpj7catp66nrwwqiuhuup,Italian Serie A | 3vo5mpj7catp66nrwwqiuhuup,INT-LAZ,3,2,4,Goal,L. Martínez,87.5,69.6
1,2878441749,41,Ball recovery,49,1,3,12,3vo5mpj7catp66nrwwqiuhuup,FC Internazionale Milano,INT,...,3vo5mpj7catp66nrwwqiuhuup,Italian Serie A | 3vo5mpj7catp66nrwwqiuhuup,PIS-INT,10,3,22,Saved Shot,L. Martínez,88.4,57.3
2,2866168927,43,Ball recovery,49,1,3,12,3vo5mpj7catp66nrwwqiuhuup,FC Internazionale Milano,INT,...,3vo5mpj7catp66nrwwqiuhuup,Italian Serie A | 3vo5mpj7catp66nrwwqiuhuup,INT-FIO,10,3,22,Saved Shot,L. Martínez,88.4,57.3
3,2890045505,57,Ball recovery,49,1,3,20,3vo5mpj7catp66nrwwqiuhuup,FC Internazionale Milano,INT,...,3vo5mpj7catp66nrwwqiuhuup,Italian Serie A | 3vo5mpj7catp66nrwwqiuhuup,UDI-INT,2,3,22,Saved Shot,L. Martínez,88.4,57.3
4,2849766989,38,Ball recovery,49,1,3,21,3vo5mpj7catp66nrwwqiuhuup,FC Internazionale Milano,INT,...,3vo5mpj7catp66nrwwqiuhuup,Italian Serie A | 3vo5mpj7catp66nrwwqiuhuup,INT-SAS,1,3,22,Saved Shot,L. Martínez,88.4,57.3


In [15]:
df_linked[["match_short", "player", "type", "shot_player", "shot_type", "dt_to_shot_sec"]].head(10)

,match_short,player,type,shot_player,shot_type,dt_to_shot_sec
0,INT-LAZ,A. Bastoni,Ball recovery,L. Martínez,Goal,3
1,PIS-INT,H. Çalhanoğlu,Ball recovery,L. Martínez,Saved Shot,10
2,INT-FIO,L. Martínez,Ball recovery,L. Martínez,Saved Shot,10
3,UDI-INT,L. Martínez,Ball recovery,L. Martínez,Saved Shot,2
4,INT-SAS,F. Esposito,Ball recovery,L. Martínez,Saved Shot,1
5,SAS-INT,L. Martínez,Ball recovery,M. Thuram,Miss,5
6,PAR-INT,H. Mkhitaryan,Ball recovery,F. Esposito,Miss,14
7,INT-LEC,M. Thuram,Ball recovery,F. Dimarco,Miss,3
8,GEN-INT,F. Esposito,Ball recovery,Y. Bisseck,Goal,9
9,INT-LEC,M. Thuram,Ball recovery,A. Bonny,Saved Shot,4


In [29]:
# 1) Ricrea SEMPRE ev da zero (così non ti porti dietro df_team vecchi)
paths = find_event_files(RESULT_DIR)
raw = load_events(paths)
ev = standardize(raw)
ev = add_phase_of_play(ev)

# 2) Controlla che nomi squadra ci sono davvero nel dataset
print("Top teams in ev:")
display(ev["team"].astype(str).value_counts().head(30))

# 3) Cerca tutte le possibili varianti di "Inter"
mask = ev["team"].astype(str).str.contains("inter", case=False, na=False)
print("Righe che matchano 'inter':", mask.sum())
display(ev.loc[mask, "team"].astype(str).value_counts().head(20))

# 4) Ora crea df_team in modo pulito
team_str = "inter"
df_team = ev.loc[mask].copy()

# (opzionale) open play
if ONLY_OPEN_PLAY:
    df_team = df_team[df_team["phase_of_play"] == "open_play"].copy()

print("Teams in df_team:", df_team["team"].unique())
print("df_team shape:", df_team.shape)

Top teams in ev:


team
Calcio Como 1907              24380
FC Internazionale Milano      24270
Juventus FC                   23967
AS Roma                       23864
SSC Napoli                    23633
Atalanta Bergamasca Calcio    22994
AC Milan                      22618
SS Lazio                      22293
Bologna FC 1909               21920
ACF Fiorentina                21266
Genoa CFC                     21000
Cagliari Calcio               20788
US Sassuolo Calcio            20526
US Cremonese                  20486
Torino FC                     20287
US Lecce                      20168
Udinese Calcio                20148
Hellas Verona FC              19933
Parma Calcio 1913             19754
Pisa Sporting Club            19366
Name: count, dtype: int64

Righe che matchano 'inter': 24270


team
FC Internazionale Milano    24270
Name: count, dtype: int64

Teams in df_team: ['FC Internazionale Milano']
df_team shape: (21144, 250)
